In [17]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import numpy as np
import scipy.stats as stats

plt.rcParams["font.family"] = "Malgun Gothic"   # 한글 깨짐 방지(윈도우 기본 폰트)
plt.rcParams["axes.unicode_minus"] = False

# df1=pd.read_csv("../data/실습데이터/2_7.CARD_SUBWAY_MONTH_202607.csv",
#                encoding='utf-8', 
#                index_col=False)#index_col=False csv 안에 열을 df 인덱스로 사용하는 것 off
# df1.tail()

import pandas as pd
from pathlib import Path

# 데이터 폴더
base_path = Path(
    r"C:\study-with-ai\src\study_with_ai\data\실습데이터"
)

#25년도 데이터
csv_files = list(base_path.glob("*CARD_SUBWAY_MONTH_2025*.csv"))
xlsx_files = list(base_path.glob("*CARD_SUBWAY_MONTH_2025*.xlsx"))




df_list = []

for file in csv_files:
    encoding = "cp949" if "202502" in file.name else "utf-8"

    df = pd.read_csv(
        file,
        encoding=encoding,
        usecols=range(6)
    )

    df_list.append(df)

df_usage = pd.concat(df_list, ignore_index=True)

df_usage

,사용일자,노선명,역명,승차총승객수,하차총승객수,등록일자
0,20250101,수인선,송도,1453,1321,20250104
1,20250101,4호선,창동,12477,13408,20250104
2,20250101,4호선,쌍문,12792,12199,20250104
3,20250101,4호선,수유(강북구청),17606,17442,20250104
4,20250101,4호선,미아(서울사이버대학),6819,6532,20250104
...,...,...,...,...,...,...
225227,20250930,일산선,정발산,8290,8421,20251003
225228,20250930,일산선,주엽,10253,10088,20251003
225229,20250930,일산선,대화,12424,9959,20251003
225230,20250930,장항선,봉명,1599,1604,20251003


In [18]:
# 인코딩 확인방법
from charset_normalizer import from_path

file = csv_files[0]

for file in csv_files:
    print(file.name, ":", from_path(file).best().encoding)

25_1.CARD_SUBWAY_MONTH_202501.csv : utf_8
25_10.CARD_SUBWAY_MONTH_202510.csv : utf_8
25_11.CARD_SUBWAY_MONTH_202511.csv : utf_8
25_12.CARD_SUBWAY_MONTH_202512.csv : utf_8
25_2.CARD_SUBWAY_MONTH_202502.csv : cp949
25_3.CARD_SUBWAY_MONTH_202503.csv : utf_8
25_4.CARD_SUBWAY_MONTH_202504.csv : utf_8
25_5.CARD_SUBWAY_MONTH_202505.csv : utf_8
25_6.CARD_SUBWAY_MONTH_202506.csv : utf_8
25_7.CARD_SUBWAY_MONTH_202507.csv : utf_8
25_8.CARD_SUBWAY_MONTH_202508.csv : utf_8
25_9.CARD_SUBWAY_MONTH_202509.csv : utf_8


In [11]:
#데이터 합치기
df_usage["사용일자"] = pd.to_datetime(
    df_usage["사용일자"].astype(str),
    format="%Y%m%d"
)

print(df_usage["사용일자"].min())
print(df_usage["사용일자"].max())

2025-01-01 00:00:00
2025-12-31 00:00:00


In [12]:
# 1~9호선만 선택
target_lines = [f"{i}호선" for i in range(1, 10)]

df_usage = df_usage[
    df_usage["노선명"].isin(target_lines)
].copy()

# 이용량 = 승차 + 하차
df_usage["이용량"] = (
    df_usage["승차총승객수"]
    + df_usage["하차총승객수"]
)



### 9호선 데이터 불러오기 및 전처리

In [13]:

path_9 = r"C:\study-with-ai\src\study_with_ai\data\실습데이터\4_2025년 9호선 역별 시간별 혼잡도 자료.xlsx"

sheet_info = {
    "상선일반(평일)": ["상선", "일반", "평일"],
    "상선일반(휴일)": ["상선", "일반", "휴일"],
    "하선일반(평일)": ["하선", "일반", "평일"],
    "하선일반(휴일)": ["하선", "일반", "휴일"],
    "상선급행(평일)": ["상선", "급행", "평일"],
    "상선급행(휴일)": ["상선", "급행", "휴일"],
    "하선급행(평일)": ["하선", "급행", "평일"],
    "하선급행(휴일)": ["하선", "급행", "휴일"]
}

df9_list = []

for sheet, info in sheet_info.items():
    temp = pd.read_excel(path_9, sheet_name=sheet, header=1)
    temp = temp.rename(columns={"구분": "역명"})
    temp["상하구분"] = info[0]
    temp["열차구분"] = info[1]
    temp["요일구분"] = info[2]
    temp["호선"] = "9호선"
    df9_list.append(temp)

df9 = pd.concat(df9_list, ignore_index=True)

df9.head()

,역명,05:30~05:59,06:00~06:29,06:30~06:59,07:00~07:29,07:30~07:59,08:00~08:29,08:30~08:59,09:00~09:29,09:30~09:59,...,22:00~22:29,22:30~22:59,23:00~23:29,23:30~23:59,00:00~00:29,00:30~00:59,상하구분,열차구분,요일구분,호선
0,개화,2.486,5.116,3.690,10.634,8.486,9.296,5.626,6.168,4.932,...,1.380,0.550,1.423333,2.398333,1.248333,1.393333,상선,일반,평일,9호선
1,김포공항,2.262,12.176,13.282,28.908,45.468,48.390,54.830,38.218,25.516,...,7.454,5.208,7.416667,7.491667,5.520000,2.450000,상선,일반,평일,9호선
2,공항시장,4.904,14.620,14.610,32.682,48.898,54.092,65.312,43.132,27.766,...,7.180,4.838,6.926667,6.855000,5.436667,2.196667,상선,일반,평일,9호선
3,신방화,12.278,19.432,21.436,39.950,60.414,66.710,75.592,53.990,33.748,...,12.580,4.496,6.826667,6.768333,5.435000,2.023333,상선,일반,평일,9호선
4,마곡나루,5.598,17.800,19.506,36.166,54.568,60.630,68.624,48.342,31.712,...,14.432,7.096,8.421667,10.473333,7.341667,2.115000,상선,일반,평일,9호선


In [14]:
time_cols = [
    col for col in df9.columns
    if "~" in str(col)
]

df9_long = df9.melt(
    id_vars=["요일구분", "호선", "역명", "상하구분", "열차구분"],
    value_vars=time_cols,
    var_name="시간대",
    value_name="혼잡도"
)

df9_long["혼잡도"] = pd.to_numeric(
    df9_long["혼잡도"],
    errors="coerce"
)

df9_long.head()

print(df9_long.shape)
print(df9_long.columns)

(8424, 7)
Index(['요일구분', '호선', '역명', '상하구분', '열차구분', '시간대', '혼잡도'], dtype='str')


In [15]:
df_usage = df_usage.rename(columns={"노선명": "호선"})

df_usage["이용량"] = (
    df_usage["승차총승객수"] +
    df_usage["하차총승객수"]
)

df_usage["역명_clean"] = (
    df_usage["역명"]
    .str.replace(r"\(.*?\)", "", regex=True)
    .str.strip()
)

usage_summary = (
    df_usage
    .groupby(["호선", "역명_clean"], as_index=False)
    .agg(평균일일이용량=("이용량", "mean"))
)
usage_summary.sort_values(by='평균일일이용량',ascending=False).head()

,호선,역명_clean,평균일일이용량
52,2호선,잠실,157200.756164
59,2호선,홍대입구,152924.520548
10,2호선,강남,151912.172603
2,1호선,서울역,139159.208219
14,2호선,구로디지털단지,106734.052055


In [30]:
import plotly.express as px

top100 = usage_summary.sort_values(
      "평균일일이용량",
      ascending=False
  ).head(100)

fig = px.bar(
    top100,
    x="역명_clean",
    y="평균일일이용량",
    color="호선",
    title="2025년 지하철 평균 일일 이용량 TOP 100",
    
)

fig.show()

### 25년도 지하철 혼잡도

In [93]:
# 기존 파일 컬럼
path_8= [
"../data/실습데이터/서울교통공사_지하철혼잡도정보_20250331.xlsx",
"../data/실습데이터/서울교통공사_지하철혼잡도정보_20250630.csv",
"../data/실습데이터/서울교통공사_지하철혼잡도정보_20251130.csv",
"../data/실습데이터/서울교통공사_지하철혼잡도정보_20260331.xlsx"
]  
df8_raw = pd.concat( #함수안에 리스트에 넣고 인덱스 번호 부여 
    [
        (
            pd.read_excel(path)
            if path.endswith(".xlsx")
            else pd.read_csv(path, encoding="cp949")
        )
        for path in path_8
    ],
    ignore_index=True
)

df8_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 5561 entries, 0 to 5560
Data columns (total 86 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   연번           1662 non-null   float64
 1   요일구분         5004 non-null   str    
 2   호선           5561 non-null   object 
 3   역번호          5561 non-null   int64  
 4   출발역          5004 non-null   str    
 5   상하구분         5561 non-null   str    
 6   5시30분        5004 non-null   float64
 7   6시00분        5004 non-null   float64
 8   6시30분        5004 non-null   float64
 9   7시00분        5004 non-null   float64
 10  7시30분        5004 non-null   float64
 11  8시00분        5004 non-null   float64
 12  8시30분        5004 non-null   float64
 13  9시00분        5004 non-null   float64
 14  9시30분        5004 non-null   float64
 15  10시00분       5004 non-null   float64
 16  10시30분       5004 non-null   float64
 17  11시00분       5004 non-null   float64
 18  11시30분       5004 non-null   float64
 19  12시00분       5004

In [82]:
#변경 컬럼
path_8= [
    "../data/실습데이터/서울교통공사_지하철혼잡도정보_20250331.xlsx",
    "../data/실습데이터/서울교통공사_지하철혼잡도정보_20250630.csv",
    "../data/실습데이터/서울교통공사_지하철혼잡도정보_20251130.csv",
    "../data/실습데이터/서울교통공사_지하철혼잡도정보_20260331.xlsx"
]
df8 = pd.concat(
      [
          (
              pd.read_excel(path)
              if path.endswith(".xlsx")
              else pd.read_csv(path, encoding="cp949")
           ).rename(columns={"출발역": "역명"})
          for path in path_8
      ],
      ignore_index=True,
      # 데이터 결합시 공통 컬럼만 남기는 옵션
  )

df8.head()
# print(df8.shape)
# print(df8.columns)



,연번,요일구분,호선,역번호,역명,상하구분,5시30분,6시00분,6시30분,7시00분,...,20:00~20:30,20:30~21:00,21:00~21:30,21:30~22:00,22:00~22:30,22:30~23:00,23:00~23:30,23:30~24:00,24:00~24:30,24:30~25:00
0,1.0,평일,1,158,청량리,상선,7.2,6.9,4.5,8.3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2.0,평일,1,157,제기동,상선,7.6,8.7,6.5,8.7,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3.0,평일,1,156,신설동,상선,6.7,11.2,7.2,9.6,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4.0,평일,1,159,동묘앞,상선,6.3,11.8,7.4,12.2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5.0,평일,1,155,동대문,상선,7.4,11.2,8.3,14.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [79]:
id_cols = ["요일구분", "호선", "역번호", "역명", "상하구분"]
# time_cols = [c for c in df8.columns if "~" in str(c)]

df8_long = df8.melt(
id_vars=id_cols,
value_vars=[c for c in df8.columns if "~" in str(c)],
var_name="시간대",
value_name="혼잡도"
)
df8_long.head(2)

df8_long["열차구분"] = "일반"

print(df8_long[["요일구분", "혼잡도"]].isna().sum())
print(df9_long[["요일구분", "혼잡도"]].isna().sum())

요일구분     21723
혼잡도     195156
dtype: int64
요일구분      0
혼잡도     216
dtype: int64


In [74]:
df8_long.tail()

,요일구분,호선,역번호,역명,상하구분,시간대,혼잡도
216874,NaN,8호선,2826,수진,하선,24:30~25:00,0.523504
216875,NaN,8호선,2827,모란,상선,24:30~25:00,0.576923
216876,NaN,8호선,2827,모란,하선,24:30~25:00,0.000000
216877,NaN,8호선,2828,남위례,상선,24:30~25:00,1.485043
216878,NaN,8호선,2828,남위례,하선,24:30~25:00,7.329060


In [ ]:
# id_cols = ["요일구분", "호선", "역번호", "역명", "상하구분"]
# time_cols = [c for c in df8.columns if "~" in c]

# df8_long = df8.melt(
#     id_vars=id_cols,
#     value_vars=time_cols,
#     var_name="시간대",
#     value_name="혼잡도"
# )

# df8_long["열차구분"] = "일반"
# df8_long["혼잡도"] = pd.to_numeric(
#     df8_long["혼잡도"], errors="coerce"
# )

In [ ]:
print(df8.columns.tolist())
print(df8.index.names)
print(set(cols) - set(df8.columns))

[None]
{'열차구분', '시간대', '혼잡도'}


In [64]:
# print(df8_long.shape)
# print(df9_long.shape)

print(df8_long.columns.tolist())
print(df9_long.columns.tolist())

# print(df8_long["요일구분"].value_counts(dropna=False))
# print(df9_long["요일구분"].value_counts(dropna=False))

# print(df8_long.isna().sum())
# print(df9_long.isna().sum())
# print(set(df8_long.columns) ==set(df9_long.columns))

['요일구분', '호선', '역명', '상하구분', '열차구분', '시간대', '혼잡도']
['요일구분', '호선', '역명', '상하구분', '열차구분', '시간대', '혼잡도']


### 1~8 호선 및 9호선 합치기

In [61]:

df8_long = df8_long.rename(
    columns={"출발역": "역명"} # 출발역을 역명으로 대체
)
print(df8_long.columns.tolist())
print(df9_long.columns.tolist())

['요일구분', '호선', '역명', '상하구분', '열차구분', '시간대', '혼잡도']
['요일구분', '호선', '역명', '상하구분', '열차구분', '시간대', '혼잡도']


In [ ]:
cols = ["요일구분", "호선", "역명", "상하구분","열차구분", "시간대", "혼잡도"]

df8_long = df8_long[cols]
df9_long = df9_long[cols]

congestion_all = pd.concat(
    [df8_long, df9_long],
    ignore_index=True
)

In [ ]:
# cols = ["요일구분", "호선", "역명", "상하구분",
#         "열차구분", "시간대", "혼잡도"]

# df8_long["상하구분"] = pd.NA
# df8_long["열차구분"] = "일반"

# df8_long = df8_long[cols]
# df9_long = df9_long[cols]

# congestion_all = pd.concat(
# [df8_long, df9_long],
# ignore_index=True
# )

In [38]:
print(congestion_all.shape)
congestion_all.columns = congestion_all.columns.str.strip()
print(
    congestion_all["호선"]
    .astype(str)
    .value_counts()
    .sort_index()
)


print(congestion_all["호선"].astype("string").value_counts(dropna=False))

(225303, 7)
호선
1       2340
1호선     5460
2      12402
2호선    29484
3       7956
3호선    18564
4       6084
4호선    14196
5      13221
5호선    30849
6       8541
6호선    20202
7       9828
7호선    22932
8       4446
8호선    10374
9호선     8424
Name: count, dtype: int64
호선
5호선    30849
2호선    29484
7호선    22932
6호선    20202
3호선    18564
4호선    14196
5      13221
2      12402
8호선    10374
7       9828
6       8541
9호선     8424
3       7956
4       6084
1호선     5460
8       4446
1       2340
Name: count, dtype: int64[pyarrow]


### 노선명 - 호선 이름 통일

In [39]:
df_usage = df_usage.rename(columns={"노선명": "호선"})

# 그 아래에 이용량과 역명 정제를 작성하세요.

df_usage["이용량"] = (
    df_usage["승차총승객수"] +
    df_usage["하차총승객수"]
)

df_usage["역명_clean"] = (
    df_usage["역명"]
    .str.replace(r"\(.*?\)", "", regex=True)
    .str.strip()
)

In [40]:
#1. 이용량 만들기

df_usage["이용량"] = (
    df_usage["승차총승객수"] +
    df_usage["하차총승객수"]
)

df_usage["요일구분"] = (
    pd.to_datetime(df_usage["사용일자"])
    .dt.dayofweek
    .map({5: "토요일", 6: "일요일"})
    .fillna("평일")
)

df_usage["역명_clean"] = (
    df_usage["역명"]
    .str.replace(r"\(.*?\)", "", regex=True)
    .str.strip()
)

In [41]:
df_usage = df_usage.rename(
    columns={"노선명": "호선"}
)

In [42]:
usage_stations = set(
    zip(df_usage["호선"], df_usage["역명"])
)

congestion_stations = set(
    zip(congestion_all["호선"], congestion_all["역명"])
)

not_matched = usage_stations - congestion_stations

print("매칭 안 된 역 수:", len(not_matched))
print(sorted(not_matched))

매칭 안 된 역 수: 391
[('1호선', '청량리(서울시립대입구)'), ('2호선', '강변(동서울터미널)'), ('2호선', '교대(법원.검찰청)'), ('2호선', '구의(광진구청)'), ('2호선', '낙성대(강감찬)'), ('2호선', '대림(구로구청)'), ('2호선', '동대문역사문화공원(DDP)'), ('2호선', '삼성(무역센터)'), ('2호선', '서울대입구(관악구청)'), ('2호선', '신촌'), ('2호선', '왕십리(성동구청)'), ('2호선', '용두(동대문구청)'), ('2호선', '잠실(송파구청)'), ('2호선', '충정로(경기대입구)'), ('3호선', '경복궁(정부서울청사)'), ('3호선', '교대(법원.검찰청)'), ('3호선', '남부터미널(예술의전당)'), ('3호선', '양재(서초구청)'), ('4호선', '당고개'), ('4호선', '동대문역사문화공원(DDP)'), ('4호선', '동작(현충원)'), ('4호선', '미아(서울사이버대학)'), ('4호선', '삼각지(전쟁기념관)'), ('4호선', '성신여대입구(돈암)'), ('4호선', '수유(강북구청)'), ('4호선', '숙대입구(갈월)'), ('4호선', '이촌(국립중앙박물관)'), ('4호선', '총신대입구(이수)'), ('4호선', '한성대입구(삼선교)'), ('4호선', '회현(남대문시장)'), ('5호선', '광나루(장신대)'), ('5호선', '광화문(세종문화회관)'), ('5호선', '군자(능동)'), ('5호선', '굽은다리(강동구민회관앞)'), ('5호선', '동대문역사문화공원(DDP)'), ('5호선', '신정(은행정)'), ('5호선', '아차산(어린이대공원후문)'), ('5호선', '오목교(목동운동장앞)'), ('5호선', '왕십리(성동구청)'), ('5호선', '천호(풍납토성)'), ('5호선', '충정로(경기대입구)'), ('5호선', '하남시청(덕풍·신장)'), ('6호선', '고려대(종암)'), ('6호선', '광흥창(서강)')

###  위와 같이 매칭 실패한 역에 대해서 역명 통일


In [43]:
df_usage["역명_clean"] = (
    df_usage["역명"]
    .str.replace(r"\(.*?\)", "", regex=True)
    .str.strip()
)

congestion_all["역명_clean"] = (
    congestion_all["역명"]
    .str.replace(r"\(.*?\)", "", regex=True)
    .str.strip()
)

In [44]:
usage_stations = set(
    zip(df_usage["호선"], df_usage["역명_clean"])
)

congestion_stations = set(
    zip(congestion_all["호선"], congestion_all["역명_clean"])
)

not_matched = usage_stations - congestion_stations

print("매칭 안 된 역 수:", len(not_matched))
print(sorted(not_matched)) 
#매칭 안 된 역 수: 3
#[('4호선', '당고개'), ('7호선', '까치울'), ('7호선', '상동')]
# 범위가 다른 경우는 통일에서 제외

매칭 안 된 역 수: 323
[('4호선', '당고개'), ('7호선', '까치울'), ('7호선', '상동'), ('9호선2~3단계', '둔촌오륜'), ('9호선2~3단계', '봉은사'), ('9호선2~3단계', '삼성중앙'), ('9호선2~3단계', '삼전'), ('9호선2~3단계', '석촌'), ('9호선2~3단계', '석촌고분'), ('9호선2~3단계', '선정릉'), ('9호선2~3단계', '송파나루'), ('9호선2~3단계', '언주'), ('9호선2~3단계', '올림픽공원'), ('9호선2~3단계', '종합운동장'), ('9호선2~3단계', '중앙보훈병원'), ('9호선2~3단계', '한성백제'), ('경강선', '경기광주'), ('경강선', '곤지암'), ('경강선', '부발'), ('경강선', '삼동'), ('경강선', '성남'), ('경강선', '세종대왕릉'), ('경강선', '신둔도예촌'), ('경강선', '여주'), ('경강선', '이매'), ('경강선', '이천'), ('경강선', '초월'), ('경강선', '판교'), ('경부선', '가산디지털단지'), ('경부선', '관악'), ('경부선', '광명'), ('경부선', '구로'), ('경부선', '군포'), ('경부선', '금정'), ('경부선', '금천구청'), ('경부선', '남영'), ('경부선', '노량진'), ('경부선', '당정'), ('경부선', '대방'), ('경부선', '독산'), ('경부선', '두정'), ('경부선', '명학'), ('경부선', '병점'), ('경부선', '서동탄'), ('경부선', '서울역'), ('경부선', '서정리'), ('경부선', '석수'), ('경부선', '성균관대'), ('경부선', '성환'), ('경부선', '세류'), ('경부선', '세마'), ('경부선', '송탄'), ('경부선', '수원'), ('경부선', '신길'), ('경부선', '신도림'), ('경부선', '안양'), ('경부선', '영등포'), ('경부선', '오산'), 

In [45]:
print(df_usage.columns.tolist())

# usage_summary = (
# df_usage
# .groupby(["호선", "역명_clean"], as_index=False)
# .agg(
#     평균일일이용량=("이용량", "mean")
# )
# )

# usage_summary.head()

['사용일자', '호선', '역명', '승차총승객수', '하차총승객수', '등록일자', '이용량', '역명_clean', '요일구분']


In [46]:
print(usage_summary.shape)

usage_summary.sort_values(
    "평균일일이용량",
    ascending=False
).head(10).round(2)

(304, 3)


,호선,역명_clean,평균일일이용량
52,2호선,잠실,157200.76
59,2호선,홍대입구,152924.52
10,2호선,강남,151912.17
2,1호선,서울역,139159.21
14,2호선,구로디지털단지,106734.05
37,2호선,신림,106059.11
26,2호선,삼성,103854.16
31,2호선,성수,102204.87
63,3호선,고속터미널,99488.98
30,2호선,선릉,97943.35


# 혼잡도 역별 구별 및 요약

In [59]:
congestion_summary = (
    congestion_all
    .groupby(["호선", "역명_clean"], as_index=False)
    .agg(
        평균혼잡도=("혼잡도", "mean"),
        최고혼잡도=("혼잡도", "max")
    )
)

congestion_summary.head()

,호선,역명_clean,평균혼잡도,최고혼잡도
0,1,동대문,NaN,NaN
1,1,동묘앞,NaN,NaN
2,1,서울역,NaN,NaN
3,1,시청,NaN,NaN
4,1,신설동,NaN,NaN


In [60]:
print(congestion_summary.shape)

congestion_summary.sort_values(
    "평균혼잡도",
    ascending=False
).head(10)

(594, 4)


,호선,역명_clean,평균혼잡도,최고혼잡도
566,9호선,동작,64.646489,185.772000
558,9호선,고속터미널,63.860283,180.134000
564,9호선,노량진,62.665651,184.806000
547,8호선,석촌,58.545308,123.564815
548,8호선,송파,58.350119,129.136752
301,2호선,방배,56.277846,143.880076
307,2호선,서초,56.160707,140.446983
290,2호선,교대,56.089094,132.846870
553,8호선,잠실,55.586219,131.657509
586,9호선,여의도,54.900559,170.644000


### 이용량 + 혼잡도 데이터 결합

In [49]:
final_df = pd.merge(
    usage_summary,
    congestion_summary,
    on=["호선", "역명_clean"],
    how="inner"
)

final_df.head()

,호선,역명_clean,평균일일이용량,평균혼잡도,최고혼잡도
0,1호선,동대문,24403.350685,32.085973,89.370921
1,1호선,동묘앞,21069.597260,30.107905,87.021997
2,1호선,서울역,139159.208219,34.940663,126.896302
3,1호선,시청,51741.136986,32.996750,113.647571
4,1호선,신설동,26976.764384,28.859670,77.344543


In [50]:
print(final_df.shape)
print(final_df.isnull().sum())

(301, 5)
호선          0
역명_clean    0
평균일일이용량     0
평균혼잡도       0
최고혼잡도       0
dtype: int64


### 평균 일일이용량

In [51]:
final_df.sort_values(
    "평균일일이용량",
    ascending=False
).head().round(2)

,호선,역명_clean,평균일일이용량,평균혼잡도,최고혼잡도
52,2호선,잠실,157200.76,34.12,93.78
59,2호선,홍대입구,152924.52,41.23,120.24
10,2호선,강남,151912.17,54.03,128.97
2,1호선,서울역,139159.21,34.94,126.90
14,2호선,구로디지털단지,106734.05,42.16,92.69


#### 시각화 산점도

In [52]:
import plotly.express as px

fig = px.scatter(
    final_df,
    x="평균일일이용량",
    y="평균혼잡도",
    hover_data=["호선", "역명_clean"],
    title="역별 평균 일일 이용량과 평균 혼잡도"
)

fig.show()

In [53]:
from scipy.stats import spearmanr

rho, p_value = spearmanr(
    final_df["평균일일이용량"],
    final_df["평균혼잡도"]
)

print("Spearman 상관계수:", round(rho, 3))
print("p-value:", p_value)

Spearman 상관계수: 0.377
p-value: 1.3074962956328197e-11


#### 상관분석결과 
- Spearman ρ = 0.362이므로 약한~중간 정도의 양의 상관관계에 있음 
- 이용객이 많은 역일수록 혼잡도가 높아지는 경향은 존재하지만, 
- 이용량만으로 혼잡도를 충분히 설명할 정도로 강한 관계는 아니다.

- 역별 평균 일일 이용량과 평균 혼잡도의 Spearman 상관분석 결과, 상관계수는 0.362로 나타났다. 
- 이는 이용량이 증가할수록 혼잡도 역시 증가하는 경향이 있으나, 그 관계는 강하지 않음을 의미한다. 
- 또한 p-value가 0.05보다 작아 해당 상관관계는 통계적으로 유의한 것으로 나타났다.

In [54]:
from scipy.stats import zscore

final_df["이용량_z"] = zscore(final_df["평균일일이용량"])
final_df["혼잡도_z"] = zscore(final_df["평균혼잡도"])

final_df["차이"] = (
    final_df["혼잡도_z"]
    - final_df["이용량_z"]
)

### zscore 사용 이유
"""
- 강남역
- 평균일일이용량 = 151,912명
- 평균혼잡도   = 43.17
- 단위가 다른 값은 계산 하기가 힘들다
"""
#### Z-score(표준점수)를 사용하여 평균에서 얼마나 떨어져 있는 지 표준편차 기준으로 표현



#### 이용량에 비해 혼잡도가 높은역

In [56]:
final_df.sort_values(
    "차이",
    ascending=False
)[
    ["호선", "역명_clean", "평균일일이용량", "평균혼잡도", "차이"]
].head(10)

,호선,역명_clean,평균일일이용량,평균혼잡도,차이
286,9호선,동작,3757.638356,64.646489,3.475414
95,4호선,남태령,2577.446575,50.309054,2.435252
267,8호선,석촌,18384.860274,58.545308,2.431167
268,8호선,송파,18547.950685,58.350119,2.409882
257,8호선,가락시장,17108.443836,54.156714,2.149172
263,8호선,몽촌토성,14519.857534,52.611390,2.134936
278,9호선,고속터미널,37205.367123,63.860283,2.085771
243,7호선,용마산,10969.769863,48.724601,1.981400
131,5호선,길동,17394.210959,51.869354,1.964376
91,3호선,충무로,4.845070,40.013794,1.756944


### 이용량은 많지만 혼잡도는 낮은 역

In [57]:
final_df.sort_values(
    "차이"
)[
    ["호선", "역명_clean", "평균일일이용량", "평균혼잡도", "차이"]
].head(10)

,호선,역명_clean,평균일일이용량,평균혼잡도,차이
52,2호선,잠실,157200.756164,34.121292,-4.940652
31,2호선,성수,102204.868493,13.191316,-4.340719
59,2호선,홍대입구,152924.520548,41.229264,-4.231670
2,1호선,서울역,139159.208219,34.940663,-4.161113
10,2호선,강남,151912.172603,54.033817,-3.220549
36,2호선,신도림,95501.717808,28.066208,-2.946329
26,2호선,삼성,103854.156164,38.805797,-2.464164
14,2호선,구로디지털단지,106734.052055,42.161879,-2.324217
37,2호선,신림,106059.112329,42.081691,-2.303458
106,4호선,상계,36500.567123,9.400421,-2.015454


### 인사이트 정리

- 이용객이 많은 역일수록 혼잡도도 높아지는 경향은 있다.
- 그 관계는 강하지 않다.
    - 사람이 많이 타는 역 = 무조건 혼잡함. 이라고 할 수는 없다 
    - ex) 배차간격, 열차 수, 환승 구조, 시간대 집중도 같은 요인이 혼잡도에 영향을 줄 가능성이 있다
- 일부 역은 전체 경향에서 크게 벗어난다
    - 성수 처럼 이용량은 높으나 평균혼잡도는 상대적으로 낮을 수 있다
    - 반대로 이용량에 비에 혼잡도가 높은 지역일 있을 수 있다는 것 이다.

- 서울 지하철은 이용객이 많은 역일수록 혼잡도가 높아지는 경향이 있지만, 그 상관은 강하지 않으며 역별 운영 특성과 이용 패턴에 따라 큰 차이가 존재한다.

In [58]:
line_summary = (
    final_df
    .groupby("호선", as_index=False)
    .agg(
        평균일일이용량=("평균일일이용량", "mean"),
        평균혼잡도=("평균혼잡도", "mean"),
        최고혼잡도=("최고혼잡도", "max")
    )
)

line_summary.sort_values(
    "평균일일이용량",
    ascending=False
)

,호선,평균일일이용량,평균혼잡도,최고혼잡도
1,2호선,55303.914247,39.036155,147.660360
0,1호선,50970.336164,30.723318,126.896302
3,4호선,39947.922460,33.365037,138.826142
2,3호선,30695.038957,32.417758,134.971447
6,7호선,26866.507958,37.266962,147.551309
4,5호선,22902.913650,30.723713,134.808917
8,9호선,22578.630795,35.770291,185.772000
7,8호선,21125.264744,33.199033,146.525641
5,6호선,17082.707560,24.589804,118.859873
